# Romer-Romer Shock

## Getting R007 (National Interbank Funding Center)

In [1]:
import os
import akshare as ak
import pandas as pd
import time
import numpy as np


os.chdir('/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data')
print("Current Directory:", os.getcwd())


Current Directory: /Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data


In [28]:
# 1. Create a list of years to loop through
years = range(2000, 2026) 
all_data = []

print("Starting download loop...")

for year in years:
    start_date = f"{year}0101"
    end_date = f"{year}1231"
    
    print(f"Fetching data for {year}...")
    
    try:
        # Fetch data for the specific year
        df = ak.repo_rate_hist(start_date=start_date, end_date=end_date)
        
        # Optional: Add a 'Year' column if you want to track it explicitly
        # df['Year'] = year 
        
        all_data.append(df)
        
        # Pause briefly to be polite to the server (prevents blocking)
        time.sleep(1) 
        
    except Exception as e:
        print(f"Error fetching data for {year}: {e}")
        # Continue to the next year even if one fails

# 2. Combine all years into one DataFrame
if all_data:
    full_repo_df = pd.concat(all_data, ignore_index=True)
    
    # 3. Clean duplicates (just in case) and Sort
    # The date column name might vary slightly; usually it is 'date' or '日期'
    # Check the column names if sort fails.
    if '日期' in full_repo_df.columns:
        full_repo_df.rename(columns={'日期': 'date'}, inplace=True)
        
    full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])
    full_repo_df.sort_values('date', inplace=True)
    
    # 4. Filter for R007 specifically
    # The column for the rate might be '收盘' (Close) or similar.
    # Often the dataframe contains multiple repo types. 
    # If the output contains a "symbol" or "code" column, filter for "R-007" or similar.
    # (Inspect full_repo_df.head() to be sure of the column name for 7-day repo)

    print("Download Complete!")
    print(full_repo_df.head())
    print(f"Total rows: {len(full_repo_df)}")
    

else:
    print("No data fetched.")


Starting download loop...
Fetching data for 2000...
Fetching data for 2001...
Fetching data for 2002...
Fetching data for 2003...
Fetching data for 2004...
Fetching data for 2005...
Fetching data for 2006...
Fetching data for 2007...
Fetching data for 2008...
Fetching data for 2009...
Fetching data for 2010...
Fetching data for 2011...
Fetching data for 2012...
Fetching data for 2013...
Fetching data for 2014...
Fetching data for 2015...
Fetching data for 2016...
Fetching data for 2017...
Fetching data for 2018...
Fetching data for 2019...
Fetching data for 2020...
Fetching data for 2021...
Fetching data for 2022...
Fetching data for 2023...
Fetching data for 2024...
Fetching data for 2025...
Download Complete!
        date  FR001  FR007  FR014  FDR001  FDR007  FDR014
0 2000-01-04    NaN   2.57    NaN     NaN     NaN     NaN
1 2000-01-05    NaN   2.56    NaN     NaN     NaN     NaN
2 2000-01-06    NaN   2.57    NaN     NaN     NaN     NaN
3 2000-01-07    NaN   2.58    NaN     NaN     N

In [29]:
full_repo_df['date'] = pd.to_datetime(full_repo_df['date'])


In [ ]:
full_repo_df['quarter'] = full_repo_df['date'].dt.to_period('Q')
china_r007_df = full_repo_df.groupby('quarter')['FR007'].mean().reset_index()
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)

print(china_r007_df.head())

  quarter     FR007
0  2000Q1  2.506311
1  2000Q2  2.398145
2  2000Q3  2.332576
3  2000Q4  2.360902
4  2001Q1  2.501951


# Getting CPI (Jin10)

In [34]:
cpi_df = ak.macro_china_cpi_yearly()


In [35]:
# Remove unnecessary columns and rename to English
cpi_df = cpi_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "cpi"})


In [36]:
# Convert date into proper format
cpi_df['date'] = pd.to_datetime(cpi_df['date'])

In [ ]:
# Remove values before 2000
cpi_df = cpi_df[cpi_df['date'].dt.year >= 2000]

In [ ]:
# Convert into quarter format
cpi_df['quarter'] = cpi_df['date'].dt.to_period('Q')

In [38]:
cpi_df = cpi_df.groupby('quarter')['cpi'].mean().reset_index()

# Cleaning GDP (Jin10)

In [39]:
gdp_df = ak.macro_china_gdp_yearly()

In [40]:
gdp_df = gdp_df[["日期", "今值"]].rename(columns={"日期": "date", "今值": "gdp"})

In [41]:
gdp_df['date'] = pd.to_datetime(gdp_df['date'])
gdp_df = gdp_df[gdp_df['date'].dt.year >= 2000]
gdp_df['quarter'] = gdp_df['date'].dt.to_period('Q')
gdp_df = gdp_df.groupby('quarter')['gdp'].mean().reset_index()

In [ ]:
gdp_df

,ngdp,rgdp,quarter,ngdp_growth,rgdp_growth
0,2444.000469,3481.713691,2000Q1,NaN,NaN
1,2473.976004,3503.331605,2000Q2,NaN,NaN
2,2503.902814,3532.980807,2000Q3,NaN,NaN
3,2557.406253,3583.202268,2000Q4,NaN,NaN
4,2634.372835,3648.582610,2001Q1,7.789375,4.792724
...,...,...,...,...,...
91,30226.716862,20989.897224,2022Q4,1.777767,2.151453
92,31175.558622,21585.587211,2023Q1,4.388015,3.962126
93,31193.856984,21730.307840,2023Q2,4.622558,5.367899
94,31530.700694,22021.077332,2023Q3,3.916370,4.962712


# Combining all three data

In [18]:
from functools import reduce


In [19]:
china_r007_df['quarter'] = china_r007_df['quarter'].astype(str)
cpi_df['quarter'] = cpi_df['quarter'].astype(str)
china_gdp_quarterly['quarter'] = china_gdp_quarterly['quarter'].astype(str)

In [20]:

# 1. Put your dataframes in a list
dfs = [china_r007_df, cpi_df, china_gdp_quarterly]

# 2. Define a merging function (Inner Join = Intersection of dates)
# This keeps only rows where 'quarter' exists in ALL dataframes
df_final = reduce(lambda left, right: pd.merge(left, right, on='quarter', how='inner'), dfs)

# 3. specific cleanup
# Set quarter as the index and sort it chronologically
df_final = df_final.set_index('quarter').sort_index()

# 4. Check the new timeframe
print(f"Start Date: {df_final.index.min()}")
print(f"End Date:   {df_final.index.max()}")


Start Date: 2000Q1
End Date:   2023Q4


In [21]:
df_final = df_final.drop(columns=['date', 'ngdp', 'rgdp'])
df_final

,FR007,cpi,ngdp_growth,rgdp_growth
quarter,,,,
2000Q1,2.506311,0.100897,NaN,NaN
2000Q2,2.398145,0.096920,NaN,NaN
2000Q3,2.332576,0.264006,NaN,NaN
2000Q4,2.360902,0.930059,NaN,NaN
2001Q1,2.501951,0.663740,7.789375,4.792724
...,...,...,...,...
2022Q4,1.959084,1.839685,1.777767,2.151453
2023Q1,2.257852,1.238188,4.388015,3.962126
2023Q2,2.100879,0.097308,4.622558,5.367899


In [22]:
df_final = df_final.reset_index()

In [23]:
df_final['quarter'] = df_final['quarter'].astype(str)

# Adding Inflation and RGDP Target

In [24]:

# 1. Define the targets (Midpoints used for ranges)
# Source: Government Work Reports
china_gdp_targets = {
    2000: 7.0, 2001: 7.0, 2002: 7.0, 2003: 7.0, 2004: 7.0,
    2005: 8.0, 2006: 8.0, 2007: 8.0, 2008: 8.0, 2009: 8.0,
    2010: 8.0, 2011: 8.0, 2012: 7.5, 2013: 7.5, 2014: 7.5,
    2015: 7.0, 
    2016: 6.75, # Range 6.5-7.0 -> Midpoint 6.75
    2017: 6.5, 
    2018: 6.5, 
    2019: 6.25, # Range 6.0-6.5 -> Midpoint 6.25
    2020: 6.0, # No official target set Set to 6.0 for calculation
    2021: 6.0,  # "Above 6.0" -> Treated as 6.0 baseline
    2022: 5.5, 
    2023: 5.0
}

china_cpi_targets = {
    2000: 2.0, 2001: 1.5, 2002: 1.5, 2003: 1.0, 2004: 3.0,
    2005: 4.0, 2006: 3.0, 2007: 3.0, 2008: 4.8, 2009: 4.0,
    2010: 3.0, 2011: 4.0, 2012: 4.0, 2013: 3.5, 2014: 3.5,
    2015: 3.0, 2016: 3.0, 2017: 3.0, 2018: 3.0, 2019: 3.0,
    2020: 3.5, 2021: 3.0, 2022: 3.0, 2023: 3.0
}

years = df_final['quarter'].str[:4].astype(int)
# 2. Extract the year from your PeriodIndex (assuming index is '2000Q1' etc.)
# If your index is named something else, change 'df_final.index'
df_final['target_gdp'] = years.map(china_gdp_targets)
df_final['target_cpi'] = years.map(china_cpi_targets)

# 4. Calculate the Gap immediately
df_final['gdp_gap'] = df_final['ngdp_growth'] - df_final['target_gdp']
df_final['cpi_gap'] = df_final['cpi'] - df_final['target_cpi']


In [25]:
df_final = df_final.reset_index()
df_final['quarter'] = df_final['quarter'].astype(str)
display(df_final[['quarter', 'target_gdp', 'gdp_gap', 'target_cpi', 'cpi_gap']].head())

,quarter,target_gdp,gdp_gap,target_cpi,cpi_gap
0,2000Q1,7.0,NaN,2.0,-1.899104
1,2000Q2,7.0,NaN,2.0,-1.903080
2,2000Q3,7.0,NaN,2.0,-1.735994
3,2000Q4,7.0,NaN,2.0,-1.069941
4,2001Q1,7.0,0.789375,1.5,-0.836260


In [26]:
df_final

,index,quarter,FR007,cpi,ngdp_growth,rgdp_growth,target_gdp,target_cpi,gdp_gap,cpi_gap
0,0,2000Q1,2.506311,0.100897,NaN,NaN,7.0,2.0,NaN,-1.899104
1,1,2000Q2,2.398145,0.096920,NaN,NaN,7.0,2.0,NaN,-1.903080
2,2,2000Q3,2.332576,0.264006,NaN,NaN,7.0,2.0,NaN,-1.735994
3,3,2000Q4,2.360902,0.930059,NaN,NaN,7.0,2.0,NaN,-1.069941
4,4,2001Q1,2.501951,0.663740,7.789375,4.792724,7.0,1.5,0.789375,-0.836260
...,...,...,...,...,...,...,...,...,...,...
91,91,2022Q4,1.959084,1.839685,1.777767,2.151453,5.5,3.0,-3.722233,-1.160315
92,92,2023Q1,2.257852,1.238188,4.388015,3.962126,5.0,3.0,-0.611985,-1.761812
93,93,2023Q2,2.100879,0.097308,4.622558,5.367899,5.0,3.0,-0.377442,-2.902692
94,94,2023Q3,1.961758,-0.064579,3.916370,4.962712,5.0,3.0,-1.083630,-3.064579


In [27]:
df_final.to_csv('romer-romer_data.csv', index=False)